#Text and Media Analytics (INFOMTMA)

## Midterm Take-Home Exam

### Deadline: Monday December 15 2025, 9:00

The midterm take-home exam for the Text and Media Analytics course consists of **two assignments** with six questions in total that require you to write working code blocks that produce results, and to write a discussion of the results. Each answer is evaluated on the quality of the code or code edits (as far as you were asked to write new code), whether the required results are produced, and the quality of the discussion, where applicable. Make sure your code is clearly structured and properly signposted. Explain via commenting each step. The scores for the questions add up to 100, with scores per question indicated next to the header of the question.

Please submit your Jupyter notebook file (`.ipynb`) via Brightspace. Do not submit a link to a Google Colab -- but keep in mind that your notebooks should run on the Colab, because that is how we will evaluate it.

# Exercise 2: Visual media analysis (30 points total)


All images for this exercise are sampled from online news articles about artificial intellgience (sources: NOS, NU.nl, New York times, Reuters). It's around 200 images - not an aweful lot but sufficient for some automated analysis and clustering. You can find the data in a ZIP folder on Brightspace under **CONTENT-> LECTURE SLIDES & MATERIALS -> IMAGES_TMA.ZIP**.

Place the ZIP in your Colab environment and execute the steps below.

The main question to that this exercise links is: *How do news media visually frame AI?*

**This Exercise has three components:**

- Enrich the dataset with features from the images, using e.g., colours, shapes, and objects.

- Cluster the images using at least two different approaches based on the features you select (e.g., K-Means, hierarchical clustering, PCA + K-Means, UMAP + DBSCAN).

- Try to label your clusters with an explanation (i.e., why cluster X should be labelled as Y?). Reflect on the process of clustering images based on their "content-features" (structure, atrributes, objects etc.). What are the main obstacles and important considerations for developing a working analytical pipeline? Where do these "inductive" approaches (using unsupervised ML) have strenghts/benefits, when do they have clear limitations?

For this exercise, you can go different paths. There is no 'perfect' clustering approach or only one correct 'solution'. It is more about critically exploring different paths and reflecting on the potentials and limitations to use data science for these sorts of questions.

In [ ]:
from google.colab import files
files.upload()

In [ ]:
!unzip IMAGES_TMA.zip -d /content/

In [ ]:
!pip -q install pillow-heif tqdm pandas opencv-python

In [ ]:
#just execute this - no other action needed for this code block!
from pathlib import Path
from PIL import Image
from pillow_heif import read_heif
import pandas as pd
import hashlib, shutil, time, zipfile
from tqdm import tqdm

#unzip and define roots
ZIP_PATH = Path("/content/IMAGES_TMA.zip")
EXTRACT_ROOT = Path("/content/IMAGES_TMA")

if not EXTRACT_ROOT.exists():
    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(EXTRACT_ROOT)
    print(f"Extracted {ZIP_PATH} to {EXTRACT_ROOT}")
else:
    print(f"Extraction folder already exists: {EXTRACT_ROOT}")

SRC_ROOT = EXTRACT_ROOT

OUT_DIR = SRC_ROOT / "processed_jpg"
OUT_DIR.mkdir(parents=True, exist_ok=True)

#outlet folder names
OUTLET_NAMES = {"NOS", "NU.NL", "NYT", "Reuters"}

print("Configured outlet names:", OUTLET_NAMES)


#file types
IMG_EXTS = {".jpg", ".jpeg", ".png", ".heic", ".webp", ".avif"}

def sha1_of_file(path, chunk_size=1<<20):
    h = hashlib.sha1()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

#convert to jpg
def convert_or_copy_to_jpg(src_path: Path, outlet: str, out_dir: Path) -> Path:
    """
    Convert/copy to JPG under processed_jpg/<outlet>/...
    Rebuilds structure starting at the outlet folder.
    """
    parts = src_path.parts
    outlet_idx = next(i for i, part in enumerate(parts) if part == outlet)
    rel_from_outlet = Path(*parts[outlet_idx:])

    out_subdir = out_dir / rel_from_outlet.parent
    out_subdir.mkdir(parents=True, exist_ok=True)
    out_path = out_subdir / (src_path.stem + ".jpg")

    ext = src_path.suffix.lower()

    if ext == ".heic":
        heif = read_heif(str(src_path))
        img = Image.frombytes(heif.mode, heif.size, heif.data, "raw")
        img = img.convert("RGB")
        img.save(out_path, "JPEG", quality=95)

    elif ext in {".jpg", ".jpeg"}:
        if not out_path.exists():
            shutil.copy2(src_path, out_path)

    else:  #png, webp, avif, etc.
        img = Image.open(src_path).convert("RGB")
        img.save(out_path, "JPEG", quality=95)

    return out_path

#include only valid images
def iter_image_files(root: Path):
    """
    Yield only real image files that:
      - live under root
      - have extension in IMG_EXTS
      - path contains an outlet name
      - skip '__MACOSX', 'processed_jpg', and '._' junk files
    """
    for p in root.rglob("*"):
        if "__MACOSX" in p.parts:
            continue
        if "processed_jpg" in p.parts:
            continue
        if p.name.startswith("._"):
            continue
        if not p.is_file():
            continue
        if p.suffix.lower() not in IMG_EXTS:
            continue

        #detect outlet from any path component
        outlet = next((part for part in p.parts if part in OUTLET_NAMES), None)
        if outlet is None:
            continue

        yield p, outlet



#process all images (no EXIF)
records = []

all_items = list(iter_image_files(SRC_ROOT))
print(f"Found {len(all_items)} candidate image files under {SRC_ROOT}")

outlet_counters = {}

for p, outlet in tqdm(all_items, desc="Converting + cataloguing"):
    try:
        #increment outlet label counter
        count = outlet_counters.get(outlet, 0) + 1
        outlet_counters[outlet] = count
        image_label = f"{outlet}_{count}"

        #convert/copy to JPG
        out_jpg = convert_or_copy_to_jpg(p, outlet, OUT_DIR)

        orig_stat = p.stat()
        jpg_stat = out_jpg.stat()

        #dimensions
        with Image.open(out_jpg) as im:
            width, height = im.size

        records.append({
            "filename": p.name,
            "stem": p.stem,
            "src_ext": p.suffix.lower(),
            "src_path": str(p),
            "jpg_path": str(out_jpg),
            "width": width,
            "height": height,
            "filesize_bytes": jpg_stat.st_size,
            "modified_time": time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(orig_stat.st_mtime)),
            "created_time": time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(orig_stat.st_ctime)),
            "sha1": sha1_of_file(out_jpg),
            "outlet": outlet,
            "image_label": image_label
        })

    except Exception as e:
        print(f"[WARN] Failed on {p}: {e}")

df = pd.DataFrame.from_records(records)
df.head()


## Question 1: Enrich the dataframe with image features (10 points)

Below are some examples for **selecting features** from the images themselves. You can choose any that you deem useful - **the below are merely suggestions!**

Also, apply **object detection** to enrich your image data. There are plenty of choices out there and we experimented with some in the seminars. What matters is how you justfy your choices and make them work. Summarise your choices and justify them (300 words max.).



### Question 2.1 - Answers

**Feature Selection and Data Enrichment**

I selected features that were directly inspired by the given examples demonstrated below in this file, and these are grayscaling, edge detection, sobel filters, color histograms, as well as faster R-CNN object detection. Instead of using these for only visualizations like the examples, I adapted them into numeric descriptors, that I can also use later in the clustering exercise.

First, I used a grayscale conversion to compute each image's average brightness. Instead of following the given example and displaying the simple grayscale version, I extracted one luminance value per image. Moving on, I used Canny edge detection, by calculating the proportion of edge pixels, giving a measure of visual complexity. Next, for the texture, I combined horizontal and vertical Sobel filters into a single gradient-magnitude sore, to show total texture strength. I also computed color histograms using 16 bins per RGB channel. Finally, I used Faster R-CNN object detection, with the help of the code of Seminar 3, by converting the detections into semantic features, like number of objects, number of people, object diversity, and average confidence of detection. These capture meaningful differences between images, like whether an image includes humans.

**Summary and Justification**

Overall, my goal was to combine semantic info from the object detection, with simple visual features, like edges. By summarizing detections into number of objects, people, diversity, and confidence, it is easier to compare images meaningfully.

In [ ]:
# Import libraries
import torch
import cv2
from tqdm import tqdm
from PIL import Image
from torchvision import transforms
from torchvision.models.detection import fasterrcnn_resnet50_fpn
import matplotlib.pyplot as plt
import numpy as np

# Use gpu if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load faster R-CNN that was pretrained on the COCO dataset (has a lot of object types)
# Thus only uses pretrained weights
model = fasterrcnn_resnet50_fpn(pretrained = True).to(device) # move device to gpu
model.eval() # putting model to evaluation mode (no dropout)

# Load object detection model (with the help of seminar 3..)
# The faster R-CNN predicts class ids and maps them to a label (0 = background, 1 = person etc.)
COCO_INSTANCE_CATEGORY_NAMES = [
    '__background__', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus',
    'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'N/A', 'stop sign',
    'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow',
    'elephant', 'bear', 'zebra', 'giraffe', 'N/A', 'backpack', 'umbrella', 'N/A', 'N/A',
    'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball',
    'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket',
    'bottle', 'N/A', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl',
    'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza',
    'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'N/A', 'dining table',
    'N/A', 'N/A', 'toilet', 'N/A', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone',
    'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'N/A', 'book',
    'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush'
]

# The model expects a torch tensor image i.o. a normal image array
transform = transforms.Compose([ # this converts 0-255 pixel values to floats between 0-1
    transforms.ToTensor(), # and rearranges shape to channels, height, width
])

# Feature Extraction Functions

# Reads an image, converts to grayscale, and returns the average brightness (0 = black, 255 = white)
# .. so that a brighter image has a higher mean
def grayscale_brightness(path):
  img = cv2.imread(path) # loads image as BGR
  gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) # converts to grayscale
  return gray.mean()

# Edge density using Canny
# Measures how many edges exist (so the more edges, the more detailed image)
def edge_density(path):
  img = cv2.imread(path, cv2.IMREAD_GRAYSCALE) # loads in grayscale
  edges = cv2.Canny(img, 100, 200) # 100,200 are the common canny thresholds, this line detectes edges
  return edges.mean() / 255.0 # normalizes values to [0, 1]

# Texture using sobel operator, a higher number means more texture
# Measures texture complexity
def sobel_texture(path):
  img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
  sobel_x = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize = 3) # Uses Sobel filters to detect changes in x and y direction
  sobel_y = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize = 3) # a small 3x3 kernel is standard choice
  magnitude = np.sqrt(sobel_x**2 + sobel_y**2) # Takes average magnitude of gradients, the strength of the textures
  return magnitude.mean()

# Computes a color histogram for each RGB channel
# Measures how the colors are distributed. in the image
def color_histogram(path, bins = 16): # each color channel is split into 16 groups, wanted a fast enough number
  img = cv2.imread(path) # loads BGR image
  img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) # converts to RGB
  hist = []

  # The bins are basically how many intervals per color we have, so the more bins the more detail
  for channel in range(3):  #0=red, 1=green, 2=blue
    h = cv2.calcHist([img], [channel], None, [bins], [0, 256]).flatten() # computes a histogram for 1 channel,
    # also from 0-255 that is 256 values, hence the range
    h = h / h.sum() # normalise so sum = 1 , so that the hist does not depend on the image size
    hist.extend(h) # add 16 values to the feats list

  return hist # returns 3 * 16 = 48 features

# Object detection features
# Runs faster r-cnn on an image
def object_detection_features(path, confthresh =0.5): # only keep predictions with probability > 50%
  img = Image.open(path).convert("RGB") # open img using PIL and convert to RGB cause that's what the model expects
  imgt = transform(img).to(device) # convert img to pytorch tensor (shape CxHxW, values 0-1 ) and move to gpu

  with torch.no_grad(): # no gradient computation
    out = model([imgt])[0] # model expectes a list of images, "out" = dictionary containing boxes, labels, scores

  # Convert confidence scores to numpy array, it's easier
  scores = out["scores"].detach().cpu().numpy()
  keep = scores > confthresh # only keep confident detections

  # Filter the labels and scores
  labels = out["labels"].detach().cpu().numpy()[keep]
  scores = scores[keep]

  num_objs = len(labels) # extracts number of detected objects
  num_ppl = np.sum(labels == 1) # extracts number of detected people (COCO class 1 = person)
  diversity = len(np.unique(labels)) if num_objs > 0 else 0 # extracts number of unique object types
  avg_conf = float(scores.mean()) if num_objs > 0 else 0 # extracts mean confidence of the detections

  return[num_objs, num_ppl, diversity, avg_conf]

  # Apply feats to all images

all_feats = [] # store the feature vector for each image

# tqdm() adds a progress bar
for _, row in tqdm(df.iterrows(), total = len(df)):
  path = row["jpg_path"] # image path

  # Computing features
  f_gray = grayscale_brightness(path)
  f_edge = edge_density(path)
  f_sobel = sobel_texture(path)
  f_hist = color_histogram(path, bins = 16) # returns 16 * 3 = 48 values
  f_obj = object_detection_features(path) # returns 4 values

  # Combine all features into a single long vector (from basic features to hist features and then obj detection ones)
  feats = [f_gray, f_edge, f_sobel] + f_hist + f_obj
  all_feats.append(feats)

# Converting the python list to a numpy array cause it's easier to index
all_feats = np.array(all_feats)

# Add features back to dataframe
# Count nr of histogram features once (48 values here)
num_hist_feats = len(color_histogram(df["jpg_path"].iloc[0], bins = 16))

# Basic features (3 cols)
df["grayscale_brightness"] = all_feats[:, 0]
df["edge_density"] = all_feats[:, 1]
df["sobel_texture"] = all_feats[:, 2]

# Add hist features
for i in range(num_hist_feats):
  df[f"hist_{i}"] = all_feats[:, 3 + i] # These start right after the 3 basic columns that were created before this

# Object detection features
df["num_objs"] = all_feats[:, 3 + num_hist_feats]
df["num_ppl"] = all_feats[:, 4 + num_hist_feats]
df["diversity"] = all_feats[:, 5 + num_hist_feats]
df["avg_conf"] = all_feats[:, 6 + num_hist_feats]

print("Done!")

# Summary table
df[["grayscale_brightness", "edge_density", "sobel_texture", "num_objs", "num_ppl", "diversity", "avg_conf"]].describe()

# Histograms
neonc = ["#FF00FF", "#00FFFF", "#39FF14"]
import matplotlib.pyplot as plt
plt.figure(figsize = (12, 4))
# Loop through 3 features and plot each one
for i, (col, c) in enumerate(zip(["num_objs", "num_ppl", "diversity"], neonc)):
  plt.subplot(1, 3, i + 1)
  df[col].hist(bins = 20, color = c, edgecolor = "black")
  plt.title(f"Distribution of {col}")
plt.tight_layout()
plt.show()

# Do images with people differ in brightness or edge density?
df["haspeople"] = (df["num_ppl"] > 0).astype(int)
df.groupby("haspeople")[["grayscale_brightness", "edge_density"]].mean()

# Correlation Heatmap of all main feats
selected = ["grayscale_brightness", "edge_density", "sobel_texture", "num_objs", "num_ppl", "diversity", "avg_conf"]
import seaborn as sns
plt.figure(figsize = (8, 6))
sns.heatmap(df[selected].corr(), annot = True, cmap = "coolwarm")
plt.title("Correlation between extracted features")
plt.show()

df.describe()


In [ ]:
# Object detection example: Visualising a random image from the dataset
row = df.sample(1).iloc[0] # selecting one random row as a series
img_path = row["jpg_path"]
img = Image.open(img_path)
plt.figure(figsize = (6, 4))
plt.imshow(img)
plt.axis("off")
plt.title("Example Image")

# Runs objects detection and draws boxes
img = Image.open(img_path).convert("RGB") # re open the image and convert to RGD (for pytorch)
imgt = transform(img).to(device) # converts the image to a pytorch tensor and movse to gpu

with torch.no_grad(): # disabling gradient calculation
  out = model([imgt])[0] # model expects a list of imgs

# Keep only detections with confidence > 50%
keep = out["scores"] > 0.5

# Extract bounding boxes and labels
boxes = out["boxes"][keep].detach().cpu().numpy()
labels = out["labels"][keep].detach().cpu().numpy()

# Convert from PIL to OpenCV format, as cv2 expects BGR
img_cv = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)

# Loops over all detected objects
for box, label in zip(boxes, labels):
  x1, y1, x2, y2 = map(int, box) # convert box coordinates to integers
  cv2.rectangle(img_cv, (x1, y1), (x2, y2), (0, 255, 0), 2) # Draws a rectangle (box) and class labels on the image
  # x1, y1 are the top left corner, x2, y2 the bottom right. Wanted the box to be green hence the (0, 255, 0)
  # 2 was just the pixel thickness
  cv2.putText(img_cv,
              COCO_INSTANCE_CATEGORY_NAMES[label], # convert label id to string
              (x1, y1-5), # put the label a bit above the box
              cv2.FONT_HERSHEY_SIMPLEX,
              0.7,
              (0, 255, 0), 2)

# Convert back to RGB so the plot displays correct colors
plt.figure(figsize = (6, 4))
plt.imshow(cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("Detected Objects")


In [ ]:
#Example 1: Greyscaling

import random
import cv2
import matplotlib.pyplot as plt

# pick one random image
row = df.sample(1).iloc[0]
img_path = row['jpg_path']

# load in colour (BGR format)
img_bgr = cv2.imread(img_path)

# greyscale version
img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

# show both
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
plt.title("Original")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(img_gray, cmap='gray')
plt.title("Greyscale")
plt.axis("off")

plt.tight_layout()
plt.show()


**Why grayscale?**

*   Simplicity: Colour adds 3 channels (R,G,B). For beginners, that triples data size with little added value if we just want shapes/textures.
*   Focus: Graffiti meaning usually comes from the marks and patterns rather than the exact colour.
*   Speed: Grayscale reduces dimensionality, making clustering and PCA faster and easier to visualise.

In [ ]:
#Example 2: Edge Detection

#pick one random image
row = df.sample(1).iloc[0]
img_path = row['jpg_path']

#load image
img_bgr = cv2.imread(img_path)
img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

#edge detection
edges = cv2.Canny(img_gray, threshold1=100, threshold2=200)

#display
plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
plt.title("Original")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(edges, cmap="gray")
plt.title("Edges (Canny)")
plt.axis("off")

plt.tight_layout()
plt.show()


**Canny Edge Detection**


*   Detects edges by finding strong intensity changes in the image.
*   Uses gradient magnitude + thresholding to highlight object outlines and contours.
*   Produces a binary edge map (white = edge, black = no edge).
*   Useful for clustering because different image types have distinct edge patterns.








In [ ]:
#Example 3: Sobel filters (texture)
sobel_x = cv2.Sobel(img_gray, cv2.CV_64F, 1, 0, ksize=3)  #horizontal gradients
sobel_y = cv2.Sobel(img_gray, cv2.CV_64F, 0, 1, ksize=3)  #vertical gradients

#magnitude of gradients = overall texture strength
sobel_mag = cv2.magnitude(sobel_x, sobel_y)

#display
plt.figure(figsize=(14,4))

plt.subplot(1,3,1)
plt.imshow(img_gray, cmap="gray")
plt.title("Greyscale")
plt.axis("off")

plt.subplot(1,3,2)
plt.imshow(sobel_x, cmap="gray")
plt.title("Sobel X (horizontal texture)")
plt.axis("off")

plt.subplot(1,3,3)
plt.imshow(sobel_y, cmap="gray")
plt.title("Sobel Y (vertical texture)")
plt.axis("off")

plt.tight_layout()
plt.show()


**Sobel Texture Features**


*   Measures texture and structure by computing intensity gradients in the x- and y-direction.
*   Produces two gradient maps: Sobel X (detects vertical edges/horizontal changes) and Sobel Y (detects horizontal edges/vertical changes).
*   The combined gradient magnitude reflects overall texture strength.
*   Good for clustering because images differ in how “textured” or “smooth” they are.




In [ ]:
#Example 3: Colour Historgrams
import cv2
import numpy as np
import matplotlib.pyplot as plt

#load image as RGB
def load_rgb(path):
    img_bgr = cv2.imread(path)
    if img_bgr is None:
        raise ValueError(f"Cannot read image: {path}")
    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

#pick one random image
row = df.sample(1).iloc[0]
img_path = row["jpg_path"]

img_rgb = load_rgb(img_path)

#compute per-channel histograms (256 bins each)
hist_size = 256
hist_range = [0, 256]

hist_r = cv2.calcHist([img_rgb], [0], None, [hist_size], hist_range)
hist_g = cv2.calcHist([img_rgb], [1], None, [hist_size], hist_range)
hist_b = cv2.calcHist([img_rgb], [2], None, [hist_size], hist_range)

#normalise (optional)
hist_r /= hist_r.sum()
hist_g /= hist_g.sum()
hist_b /= hist_b.sum()

#visualise: original image + histograms
fig, (ax_img, ax_hist) = plt.subplots(1, 2, figsize=(12, 4))

#left: image
ax_img.imshow(img_rgb)
ax_img.set_title("Random Image")
ax_img.axis("off")

#right: RGB histograms
bins = np.arange(hist_size)

ax_hist.plot(bins, hist_r, label="Red channel")
ax_hist.plot(bins, hist_g, label="Green channel")
ax_hist.plot(bins, hist_b, label="Blue channel")
ax_hist.set_xlim([0, 255])
ax_hist.set_xlabel("Pixel value (0–255)")
ax_hist.set_ylabel("Normalised frequency")
ax_hist.set_title("Colour histogram (RGB)")
ax_hist.legend()

plt.tight_layout()
plt.show()


**Colour Histograms**


*   Summarise an image based on the distribution of colours (how often each colour appears).
*   Typically computed in RGB (or HSV) using a fixed number of bins per channel (e.g. 8×8×8 = 512 values).
Produces a compact numerical fingerprint of the image’s overall colour palette.
*   Useful for clustering because images with similar dominant colours (e.g. blue screens, orange robotics labs, dark server rooms) naturally group together.





## Question 2: Cluster the images (10 points)

Now that you have selected your features, **try clustering the images with at least two clustering techniques of your choice** (e.g., K-Means, hierarchical clustering, PCA + K-Means, UMAP + DBSCAN).

**Try different cluster solutions** as well and, where applicable, explain/motivate your choice for the number of clusters.

**Present examples for some of the clusters** (what do the images look like in a given cluster?). You don't need to show examples for all clusters across the min. 2 techniques but can choose yourself which one you want to show.

**Discuss why you chose the clustering techniques** that you eventually applied - what considerations guided your choices? (300 words max).



### Question 2.2 - Answers

Extracting the features from all images in the previous exercise, was the main foundation to apply different clustering techniques. I selected PCA and K-Means, and, UMAP and DBSCAN, because both of them offer different prespectives regarding the clusters, which is useful. K-Means searches for centroid based clusters, while DBSCAN focuses on dense regions.

**PCA and K-Means**

Initially, I only applied K-Means. But after realising that my selected features had more than 50 dimensions (mainly due to the color histograms), I applied PCA, selecting 95% explained variance. In order to choose the number of clusters, I used the elbow method for a range of clusters from 2 to 9, and saw that after k = 5 there is a steep decrease in improvements, leading me to chooose it.

**UMAP and DBSCAN**

I used UMAP to project the image features into a 2D representation. I set n_neighbors = 30, which controls how many nearby points UMAP considers for the embeddings. A very low value will lead to many small, fragmented groups, whereas a big one will have a more "global" structure. I set min_dist = 0 because I wanted to get tight clusters that have similar images, which is beneficial since DBSCAN clusters based on density. After applying DBSCAN, I set min_samples = 5 (after tuning), which determines the minimum number of points to create a dense region. In order to choose the epsilon, which defines the radius between neighbors to determine density, I created a k-distance plot (k = min_samples). The distance from each point to its kth nearest neighbor is calculated, which gives a curve that shows points within dense clusters (flat part), and outliers (steep). The point right before the curve started steeply rising showed the best eps.

In [ ]:
# ** Question 2.2
# 1st Clustering Method: K-MEANS
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from PIL import Image
import umap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Selecting features for clustering. This totally has more than 50 dimensions, so will do PCA later
featscols = [
    "grayscale_brightness",
    "edge_density",
    "sobel_texture",
    "num_objs",
    "num_ppl",
    "diversity",
    "avg_conf"
] + [f"hist_{i}" for i in range(num_hist_feats)]

# Extracting the selected features into a NumPy array because it's faster
X = df[featscols].values

# Standardizing, cause the features have different values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Run PCA to reduce dimensionality and make clustering easier
pca = PCA(n_components = 0.95, random_state = 42) # chose to keep as many components as needed to explain 95% of the variance
X_pca = pca.fit_transform(X_scaled) # new reduced feature matrix

# PCA explained variance
# The plot shows how much total variance is captured as new components are added
plt.figure(figsize = (8, 4))
plt.plot(np.cumsum(pca.explained_variance_ratio_), marker = "o") # marker adds small round dot at each data point
plt.xlabel("Number of components")
plt.ylabel("Cumulative explained variance")
plt.title("PCA Explained Variance")
plt.grid(True)
plt.show()

# Elbow method to find optimal number of clusters (ended up being 5, because after 5 improvements are very small)
inertias = [] # total distance of points to their cluster centers (interested where it deeply drops)
K = range(2, 10) # testing cluster counts from 2 to 9

for k in K:
  kmeans = KMeans(n_clusters = k, random_state = 42)
  kmeans.fit(X_pca) # clustering the PCA output
  inertias.append(kmeans.inertia_) # store inertia val

plt.figure(figsize = (8, 4))
plt.plot(K, inertias, marker = "o")
plt.xlabel("Number of clusters")
plt.ylabel("Inertia")
plt.title("Elbow Method for Optimal K")
plt.grid(True)
plt.show()


# K-MEANS with 5 clusters
kmeans_pca = KMeans(n_clusters = 5, random_state = 42)
df["k_means_pca_cluster"] = kmeans_pca.fit_predict(X_pca) # assign each image to a cluster (from 0 to 4)

# Selecting dominant clusters to present
clusterstoshow = [1, 2, 4]

# Visualizing only the selected ones
for cluster_id in clusterstoshow:
  print("\nCLUSTER", cluster_id)

  # Selecting only the images belonging to this cluster
  sub = df[df["k_means_pca_cluster"] == cluster_id]

  # Showing max 8 images per cluster so it does not get too messy
  n = min(8, len(sub))
  sample = sub.sample(n)

  # Creating 2x4 grid of image plots since I'm using max 8 images per cluster and 2x4=8 looks best on the screen
  # ..  after trying other nr of rows x cols
  fix, axs = plt.subplots(2, 4, figsize = (12, 6))
  axs = axs.flatten()

  # Displaying the images
  for ax, (_, row) in zip(axs, sample.iterrows()):
    img = Image.open(row["jpg_path"]) # reading the image
    ax.imshow(img) # showing it
    ax.axis("off") # hiding axes ticks
    ax.set_title(f"{cluster_id}") # also show cluster id above every image so it looks organized

  plt.show()


In [ ]:
# ** Question 2.2

# 2nd Clustering Method: UMAP + DBSCAN

# Importing libraries
import umap
from sklearn.cluster import DBSCAN
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.neighbors import NearestNeighbors

# Applying UMAP to reduce the high dimensionality feature vectors into 2D representation
# because the goal is to visualize clusters on a 2D scatter plot
umap_model = umap.UMAP(
    n_neighbors = 30, # controls how much local structure umap pays attention to. I chose 30
                      # .. so that it captures more structure
    min_dist = 0.0,   # controls how tightly umap packs points together. Chose 0 to get very tight clusters
                      # as umap increases density, there will be better DBSCAN separation
    random_state = 42,
    metric = "euclidean" # using euclidean distance to measure similarity between points
)

# Fit umap and transform scaled features to 2d points
X_umap = umap_model.fit_transform(X_scaled)

# Finding a good eps value for DBSCAN using k-distance graph
# A wrong eps could lead to too many clusters or a lot of noise
k = 5

# Fit a NearestNeighbors model to umap points
nbrs = NearestNeighbors(n_neighbors = k).fit(X_umap)

# The distances become an array of distances to each of the 5 NNs
distances, indices = nbrs.kneighbors(X_umap)

# Extract the distance to the kth neighbor (k-1) and sort them to get a curve shape
k_distances = np.sort(distances[:, k - 1])

# Plot the k distance graph to decide the EPS value
# Looking for the elbow in this curve to find it
# It ended up being eps = 0.45-0.50 because right after 0.45 the curve steeply increases
plt.plot(k_distances)
plt.title("K-distance Graph for EPS Selection")
plt.show

# Will run DBSCAN clustering on the umap coords
# Store them in the df for later plotting
df["umap_x"] = X_umap[:, 0]
df["umap_y"] = X_umap[:, 1]

db = DBSCAN(
    eps = 0.48, # could be anything between 0.45-0.50 according to the elbow in the prior plot
                # points within a radius of the eps value may form a cluster
    min_samples = 5 # means a point needs at least 5 neighborswithing 0.48 radius to count as "dense"
).fit(X_umap)

# Labels found by DBSCAN (-1 is the noise, rest are cluster ids)
df["dbscan_cluster"] = db.labels_

# Showing the dominant cluster 1
# (will not explain as thoroughly, the following part is more or less copy pasted from the previous code chunk)
clustertoshow = [1]
for cluster_id in clustertoshow:
  print(f"\nCLUSTER {cluster_id}")

  sample = df[df["dbscan_cluster"] == cluster_id].sample(min(8, sum(df["dbscan_cluster"] == cluster_id)))
  fig, axes = plt.subplots(2, 4, figsize = (12, 6))
  axes = axes.flatten()

  for ax, (_, row) in zip(axes, sample.iterrows()):
    img = Image.open(row["jpg_path"])
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(f"{cluster_id}")

**ANSWER: Provide a short summary and justification of your choices here!**

## Question 3: Try Labelling the Clusters and Discussion (10 points)

**Try to label your clusters** with an explanation (i.e., why cluster X should be labelled as Y?). Present your results - if you have too many clusters, make a selection of 3-5 that you want to show.

**Provide some descriptive analyses of your clusters** (e.g., distribution, what outlets they relate to, representative images per cluster).

**Reflect on the process of clustering images based on their "content-feature**s" (structure, atrributes, objects etc.). What are the main obstacles and important considerations for developing a working analytical pipeline? Where do these "inductive" approaches (using unsupervised ML) have strenghts/benefits, when do they have clear limitations? (500 words max.)

### Question 2.3 - Answers

**Labelling the Clusters and Descriptive Analyses**

***PCA and K-Means***

Cluster 1, 2 include the most images, followed by cluster 4. Cluster 3 has the fewest, and cluster 0 only has 1 image, which is most likely an outlier. The outlet distribution heatmap shows that clusters do not depend on the outlet. All publishers (NOS, NU.nl, NYT, Reuters) are present across most clusters (except cluster 1, but it contains only 1 image so it makes sense). This shows that clustering is driven by actual visual content.

Cluster 0: "Outlier"

Cluster 1: images with moderate brightness and texture, balanced colors. Mostly sceneries, rare presence of people. Label : "Sceneries and Minimal Object Presence"

Cluster 2: intense images, some with neon colors / contrast, high density and sobel texture. Often presence of people, and less objects. Label; "Human Focused, Intense Images"

Cluster 3: mostly white backgrounds, discrete colors. Low object diversity, rare humans noted, low edge density and texture. A lot of logos, drawed images, brandings. Label: "Calm, Logos Visuals"

Cluster 4: diverse colors, some AI-focused images, high edge density and texture, political scenes, high object count (mostly humans). Label: "Complicated, High Human Presence"


***UMAP and DBSCAN***

The results show cluster 0, the main and largest cluster, cluster 1 a secondary small cluster, and cluster -1 which only contains about 2 outlier points. Cluster 0 dominates all news outlets, and cluster 1 is present but small.

Cluster -1: "Outliers"

Cluster 0: a lot of objects, both people and items, rich colors, high edge density and texture. Label: "Visual Richness and Object Diversity"

Cluster 1: mostly white backgrounds, logos, company names, low edge density and texture, moderate object counts, calm colors. Label: "Abstract, Brand Visuals".

**Reflection**

My main obstacles were that images are hard to interpret. The techniques only focus on the appearance, but of course do not have access to the semantics, the intention behind them, if they are satire or not. So it is mostly surface level. The main considerations would be to use dimensionality reduction because the features can be high dimensional and then distance becomes meaningless, following the clusters being unreliable. Additonally, to apply different methods, because there is not a single best one. Every method offers different insights. Also, I only showed a sample of the cluster contents, so the labels cannot be factual, so we should not falsely claim them to be 100% accurate.

K-Means was easier to interpret, because it forces all images into different clusters. However, it does not isolate outliers, and it assumes that all clusters have a similar size and shape, which could be false. DBSCAN, applied after UMAP, isolated the outliers and does not falsely assign a point to a cluster. But, this method can clump multiple images into one big cluster, even if they are diverse, so it is harder to interpret. That can happen because UMAP’s dimensionality reduction could oversimplify the structure.

Overall, I prefer K-Means in terms of interpretation and DBSCAN for outlier detection.

In [ ]:
# ** Question 2.3

# 1st: K-MEANS

# 1. Visualizing multiple images per cluster to be able to label and interpret them
for cluster_id in sorted(df["k_means_pca_cluster"].unique()):
  print("\nCLUSTER", cluster_id)

  # Selecting only the images belonging to this cluster
  sub = df[df["k_means_pca_cluster"] == cluster_id]

  # Showing max 8 images per cluster so it does not get too messy
  n = min(8, len(sub))
  sample = sub.sample(n)

  # Creating 2x4 grid of image plots since I'm using max 8 images per cluster and 2x4=8 looks best on the screen
  # ..  after trying other nr of rows x cols
  fix, axs = plt.subplots(2, 4, figsize = (12, 6))
  axs = axs.flatten()

  # Displaying the images
  for ax, (_, row) in zip(axs, sample.iterrows()):
    img = Image.open(row["jpg_path"]) # reading the image
    ax.imshow(img) # showing it
    ax.axis("off") # hiding axes ticks
    ax.set_title(f"{cluster_id}") # also show cluster id above every image so it looks organized

  plt.show()

# 2. Distribution of clusters (counts)
kmean_counts = df["k_means_pca_cluster"].value_counts().sort_index()
print("K-Means Cluster Counts:")
print(kmean_counts)

# Visualizing them
# Color palette
colors = ["#FF00FF", "#00FFFF", "#8A2BE2", "#FF1493", "#7DF9FF", "#00FF00", "#77DD77"]
plt.figure(figsize = (8, 4))
sns.barplot(x = kmean_counts.index, y = kmean_counts.values, palette = colors)
plt.title("K-Means Cluster Counts")
plt.xlabel("Cluster ID")
plt.ylabel("Number of Images")


# 3. Distribution of outlets per cluster
k_outlet = pd.crosstab(df["k_means_pca_cluster"], df["outlet"])

# Visualization
plt.figure(figsize = (8, 4))
sns.heatmap(k_outlet, annot = True, fmt = "d", cmap = "cool", cbar = True)
plt.title("K-Means Cluster Outlet Distribution")
plt.xlabel("Outlet")
plt.ylabel("Cluster ID")
plt.show()

# 2nd: UMAP + DBSCAN

# 1. Visualizing multiple images per cluster to be able to label and interpret them
# ** The following code is mostly copy pasted from the code right before this
# ** So no point in re explaining every step..
# Show sample images
for cluster_id in sorted(df["dbscan_cluster"].unique()):
  print(f"\nCLUSTER {cluster_id}")

  sample = df[df["dbscan_cluster"] == cluster_id].sample(min(8, sum(df["dbscan_cluster"] == cluster_id)))
  fig, axes = plt.subplots(2, 4, figsize = (12, 6))
  axes = axes.flatten()

  for ax, (_, row) in zip(axes, sample.iterrows()):
    img = Image.open(row["jpg_path"])
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(f"{cluster_id}")

# 2. Distribution of clusters (counts and percentages)
dbscan_counts = df["dbscan_cluster"].value_counts().sort_index()
print("DBSCAN Cluster Counts:")
print(dbscan_counts)

# Visualizing them
# Color palette
colors = ["#FF00FF", "#00FFFF", "#8A2BE2", "#FF1493", "#7DF9FF", "#00FF00", "#77DD77"]
plt.figure(figsize = (8, 4))
sns.barplot(x = dbscan_counts.index, y = dbscan_counts.values, palette = colors)
plt.title("DBSCAN Cluster Counts")
plt.xlabel("Cluster ID")
plt.ylabel("Number of Images")


# 3. Visualizing DBSCAN cluster on the umap plot
plt.figure(figsize = (10, 7))

# Each point is an image plotted at its umap coordinate
scatter = plt.scatter(
    df["umap_x"],
    df["umap_y"],
    c = df["dbscan_cluster"], # cluster id determines color
    cmap = "tab10", # just a color palette
    s = 10 # selecting small dots for aesthetics
)
plt.title("UMAP & DBSCAN Clusters")
plt.axis("off")
# Adding legend to explain which color means which cluster
plt.legend(*scatter.legend_elements(), title = "Cluster")
plt.show()

# 4. Distribution of outlets per cluster
d_outlet = pd.crosstab(df["dbscan_cluster"], df["outlet"])

# Visualization
plt.figure(figsize = (8, 4))
sns.heatmap(d_outlet, annot = True, fmt = "d", cmap = "cool", cbar = True)
plt.title("DBSCAN Cluster Outlet Distribution")
plt.xlabel("Outlet")
plt.ylabel("Cluster ID")
plt.show()